# Interactive 2: Finding pain in the brain with the general linear model

Same subject, same run. Now we ask: *which voxels go up and down with the heat stimulus?* Run the cells in order; lines marked `# <-- change` are the ones to play with.

In [ ]:
#@title Setup: install packages and download the data (run this first, takes about a minute)
import os, sys
SUBJECT = 'cbp014'      # <-- change: 'cbp001', 'cbp006', 'cbp014' (chronic back pain) or 'healthy007'

DATA_URL = 'https://github.com/cmahlen/fmri-pain-class/releases/download/data-v1'   # one ~160 MB tarball per subject, one 10-minute run each

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    get_ipython().system('pip install -q --no-deps nilearn ipyniivue anywidget psygnal')   # only what Colab lacks; --no-deps keeps Colab's pandas/requests
    from google.colab import output; output.enable_custom_widget_manager()               # lets the clickable brain viewer render in Colab
    DATA = f'data/{SUBJECT}'
    if not os.path.exists(DATA):
        os.makedirs('data', exist_ok=True)
        get_ipython().system(f'curl -sL {DATA_URL}/{SUBJECT}.tar.gz | tar xz -C data')
else:
    DATA = os.path.join(os.environ.get('FMRI_DATA', 'drive_upload'), SUBJECT)

import numpy as np, nibabel as nib, matplotlib.pyplot as plt, pandas as pd, warnings
from nilearn import plotting, image, masking
from nilearn.datasets import load_mni152_template
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 90
TR = 2.5                                   # seconds between volumes
t = np.arange(240) * TR                    # time axis in seconds
print('files:', sorted(os.listdir(DATA)))

## 1. What happened during the scan

In [ ]:
bold = nib.load(f'{DATA}/bold_mni.nii.gz')            # preprocessed run (motion corrected, registered to MNI, 3 mm voxels)
stim = np.loadtxt(f'{DATA}/stimulus.txt')               # probe temperature (C) at each volume
rating = np.loadtxt(f'{DATA}/rating.txt')               # subject's pain rating (0-100) at each volume
motion = np.loadtxt(f'{DATA}/motion_params.txt')

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].plot(t, stim, color='firebrick');  ax[0].set_ylabel('temperature (C)')
ax[1].plot(t, rating, color='k');        ax[1].set_ylabel('pain rating (0-100)'); ax[1].set_xlabel('time (s)')
plt.show()

## 2. From stimulus to expected BOLD signal: the hemodynamic response function

In [ ]:
from nilearn.glm.first_level.hemodynamic_models import spm_hrf

STIM_THRESHOLD = 42      # C  <-- change: temperature above which we call it "stimulus on" (baseline is 40 C, peaks 48-53 C; try 45 or 48)

hrf_fine = spm_hrf(0.1, oversampling=1, time_length=30)
hrf = spm_hrf(TR, oversampling=1, time_length=30)
boxcar = (stim > STIM_THRESHOLD).astype(float)
assert boxcar.sum() > 0, f'no volume is above {STIM_THRESHOLD} C; the hottest is {stim.max():.1f} C, lower STIM_THRESHOLD'
expected = np.convolve(boxcar, hrf)[:240]

fig, ax = plt.subplots(3, 1, figsize=(13, 8))
ax[0].plot(np.arange(len(hrf_fine)) * 0.1, hrf_fine, color='purple'); ax[0].set(title='hemodynamic response to a brief event', xlabel='seconds after event')
ax[1].plot(t, boxcar, color='firebrick'); ax[1].set(title='stimulus on / off', ylim=(-0.1, 1.2))
ax[2].plot(t, expected, color='purple');  ax[2].set(title='stimulus convolved with the HRF = expected BOLD signal', xlabel='time (s)')
plt.tight_layout(); plt.show()

## 3. The GLM in one voxel: how well does the expected signal explain the measured one?

In [ ]:
from nilearn.maskers import NiftiSpheresMasker
from nilearn.glm.first_level import make_first_level_design_matrix

COORDS = (40, 8, -2)      # <-- change: MNI coordinate (x -60..60, y -90..60, z -40..70). (40, 8, -2) right insula; (16, 10, -8) right accumbens; (-38, -22, 56) left motor cortex

def design(regressors, high_pass=0.01):
    return make_first_level_design_matrix(t, add_regs=pd.DataFrame(regressors, index=t), drift_model='cosine', high_pass=high_pass)

dm = design({'pain': expected})
y = NiftiSpheresMasker([COORDS], radius=4).fit_transform(bold)[:, 0]
y = 100 * (y - y.mean()) / y.mean()                                  # percent signal change

X = dm.values
beta = np.linalg.lstsq(X, y, rcond=None)[0]
fit = X @ beta
resid = y - fit
i = list(dm.columns).index('pain')
se = np.sqrt(resid.var(ddof=X.shape[1]) * np.linalg.inv(X.T @ X)[i, i])
print(f'beta for pain = {beta[i]:.3f} % signal change     t = {beta[i] / se:.2f}')

fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax[0].plot(t, y, color='gray', label='measured'); ax[0].plot(t, fit, color='purple', lw=2, label='GLM fit'); ax[0].legend(); ax[0].set_title(f'voxel at MNI {COORDS}')
ax[1].plot(t, resid, color='gray'); ax[1].set_title('residual (what the model does not explain)'); ax[1].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

## 4. The design matrix: the same model, written as columns

In [ ]:
INCLUDE_MOTION = False      # <-- change: add the 6 head-motion parameters as nuisance regressors

regs = {'pain': expected}
if INCLUDE_MOTION:
    regs.update({f'motion{i}': motion[:, i] for i in range(6)})
dm = design(regs)
plotting.plot_design_matrix(dm)
plotting.show()

## 5. The GLM in every voxel

In [ ]:
from nilearn.glm.first_level import FirstLevelModel

SMOOTHING = 6       # mm  <-- change: 0, 4, 6, 8
THRESHOLD = 3.1     # z   <-- change

model = FirstLevelModel(t_r=TR, smoothing_fwhm=SMOOTHING, minimize_memory=False).fit(bold, design_matrices=dm)
zmap = model.compute_contrast('pain')

plotting.plot_stat_map(zmap, threshold=THRESHOLD, cut_coords=COORDS, title=f'pain > rest, z > {THRESHOLD}, smoothing {SMOOTHING} mm')
plotting.plot_glass_brain(zmap, threshold=THRESHOLD, colorbar=True, plot_abs=False, display_mode='lyrz')
plotting.show()

In [ ]:
# Click on the map: the plot underneath shows that voxel's data and its GLM fit
from ipyniivue import NiiVue, SliceType
from ipywidgets import Output
from IPython.display import display, clear_output

mni = load_mni152_template(resolution=2); mni.to_filename('mni.nii.gz'); zmap.to_filename('zmap.nii.gz')
bold_smooth = image.smooth_img(bold, SMOOTHING).get_fdata()
nv = NiiVue(slice_type=SliceType.MULTIPLANAR, height=450)
nv.load_volumes([{'path': 'mni.nii.gz', 'colormap': 'gray'},
                 {'path': 'zmap.nii.gz', 'colormap': 'warm', 'opacity': 0.8, 'cal_min': THRESHOLD, 'cal_max': 8}])
nv.opts.is_colorbar = True
out = Output()

zmap_data = zmap.get_fdata()

@nv.on_location_change
def show_fit(loc):
    i, j, k = np.round(np.linalg.inv(bold.affine) @ [*loc['mm'][:3], 1])[:3].astype(int)
    if not (0 <= i < bold_smooth.shape[0] and 0 <= j < bold_smooth.shape[1] and 0 <= k < bold_smooth.shape[2]):
        return                                               # clicked outside the image
    y = bold_smooth[i, j, k]
    if y.mean() == 0:
        return                                               # clicked outside the brain
    y = 100 * (y - y.mean()) / y.mean()
    b = np.linalg.lstsq(dm.values, y, rcond=None)[0]
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(t, y, color='gray', label='measured'); ax.plot(t, dm.values @ b, color='purple', lw=2, label='GLM fit')
    ax.set(title=f'MNI {tuple(np.round(loc["mm"][:3]).astype(int))}    z = {zmap_data[i, j, k]:.2f}', xlabel='time (s)'); ax.legend(loc='upper right')
    plt.close(fig)
    with out:
        clear_output(wait=True); display(fig)

display(nv, out)

In [ ]:
# The same map as a self-contained interactive viewer (no clicking into Python, but works everywhere)
plotting.view_img(zmap, threshold=THRESHOLD, cut_coords=COORDS, title='pain > rest')

## 6. Multiple comparisons: how many voxels did we just test?

In [ ]:
from nilearn.glm import threshold_stats_img

n_voxels = int(model.masker_.mask_img_.get_fdata().sum())
print('voxels tested:', n_voxels, '   expected false positives at p < 0.05 if nothing is happening:', round(0.05 * n_voxels))

for label, kw in [('uncorrected p < 0.05', dict(alpha=0.05, height_control='fpr')),
                  ('uncorrected p < 0.001', dict(alpha=0.001, height_control='fpr')),
                  ('FDR corrected q < 0.05', dict(alpha=0.05, height_control='fdr')),
                  ('Bonferroni corrected p < 0.05', dict(alpha=0.05, height_control='bonferroni'))]:
    thr_map, thr = threshold_stats_img(zmap, **kw)
    plotting.plot_glass_brain(thr_map, colorbar=True, plot_abs=False, display_mode='lyrz', title=f'{label}  (z > {thr:.2f})')
plotting.show()

## 7. Two models of the same data: stimulus temperature vs perceived pain

In [ ]:
rating_expected = np.convolve(rating - rating.min(), hrf)[:240]

for name, reg in [('stimulus', expected), ('rating', rating_expected)]:
    m = FirstLevelModel(t_r=TR, smoothing_fwhm=SMOOTHING).fit(bold, design_matrices=design({name: reg}))
    plotting.plot_stat_map(m.compute_contrast(name), threshold=THRESHOLD, cut_coords=COORDS, title=f'model: {name}')
plotting.show()

## 8. The nucleus accumbens: responding to change, not to heat
Baliki et al. found the accumbens responds when the stimulus *starts* and when it *ends*. In healthy subjects both responses are positive (BOLD follows |d stim/dt|). In chronic back pain the offset response flips sign (BOLD follows d stim/dt).

In [ ]:
nac_mask = image.math_img('img > 0', img=f'{DATA}/nac_mask_mni.nii.gz')
nac = masking.apply_mask(image.smooth_img(bold, 4), image.resample_to_img(nac_mask, bold, interpolation='nearest')).mean(1)
nac = 100 * (nac - nac.mean()) / nac.mean()
D = design({}).values                                   # drift-only design matrix
nac = nac - D @ np.linalg.lstsq(D, nac, rcond=None)[0]   # remove slow drift

dstim = np.gradient(stim)
deriv = np.convolve(dstim, hrf)[:240]                 # d stim / dt   (positive at onset, negative at offset)
rect  = np.convolve(np.abs(dstim), hrf)[:240]         # |d stim / dt| (positive at onset AND offset)

fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
ax[0].plot(t, nac, color='k');            ax[0].set_title('nucleus accumbens BOLD (% signal change)')
ax[1].plot(t, deriv, color='firebrick');  ax[1].set_title(f'd stim/dt convolved with HRF      correlation with NAc = {np.corrcoef(nac, deriv)[0, 1]:.2f}')
ax[2].plot(t, rect, color='seagreen');    ax[2].set_title(f'|d stim/dt| convolved with HRF     correlation with NAc = {np.corrcoef(nac, rect)[0, 1]:.2f}'); ax[2].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

In [ ]:
# Average the accumbens signal around every stimulus onset and every stimulus offset
WINDOW = (-10, 30)       # seconds before and after the event  <-- change (keep the first number smaller than the second)

onsets  = np.where(np.diff(boxcar) == 1)[0] + 1
offsets = np.where(np.diff(boxcar) == -1)[0] + 1
lags = np.arange(int(WINDOW[0] / TR), int(WINDOW[1] / TR))

def locked(events, series):
    return np.array([series[e + lags] for e in events if e + lags[0] >= 0 and e + lags[-1] < 240])

fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for a, ev, name in [(ax[0], onsets, 'stimulus ONSET'), (ax[1], offsets, 'stimulus OFFSET')]:
    seg = locked(ev, nac)
    if len(seg) == 0:
        a.set_title(f'no complete {name} window in this run'); continue
    a.plot(lags * TR, seg.T, color='lightgray')
    a.plot(lags * TR, seg.mean(0), color='k', lw=3, label=f'mean of {len(seg)} events')
    a.axvline(0, ls='--', color='firebrick'); a.axhline(0, color='gray', lw=0.5); a.legend(); a.set(title=f'NAc around {name}', xlabel='seconds from event')
plt.show()